# Building MCP Servers

Practice writing tool functions, JSON schemas, and dispatchers against an offline, in-memory paper dataset. No API key required.

***Summary***
1. [Setup - in-memory paper store](#setup)
2. [count_papers - function + schema](#count-papers)
3. [filter_by_author - function + schema](#filter-by-author)
4. [Extending the dispatcher](#dispatcher)
5. [Defensive dispatch](#defensive-dispatch)
6. [Generalization - wrap a real API](#generalization)


***
<a id='setup'></a>
## 1. Setup - in-memory paper store

The lesson notebook uses live arXiv results, which change every day and require network access. Homework should be reproducible offline, so we substitute a fixed in-memory dictionary. Every exercise below reads from this store.


In [ ]:
import json
from typing import List

FAKE_PAPERS = {
    "2501.11111": {"title": "Attention is All You Need", "authors": ["Vaswani", "Shazeer"], "year": 2017},
    "2501.22222": {"title": "BERT: Pre-training of Deep Bidirectional Transformers", "authors": ["Devlin", "Chang"], "year": 2019},
    "2501.33333": {"title": "GPT-3: Language Models are Few-Shot Learners", "authors": ["Brown", "Mann"], "year": 2020},
    "2501.44444": {"title": "Chain-of-Thought Prompting", "authors": ["Wei", "Wang"], "year": 2022},
    "2501.55555": {"title": "Retrieval-Augmented Generation", "authors": ["Lewis", "Perez"], "year": 2020},
}

print(f"Loaded {len(FAKE_PAPERS)} papers")


***
<a id='count-papers'></a>
## 2. `count_papers` - function + schema

Warm-up exercise: implement a tool that reports how many papers are stored, optionally filtered by year, then write its schema.


**Q1 a) Implement `count_papers(min_year: int = 0) -> int`. Return the number of papers with `year >= min_year`. Store your function so the tests below can reach it.**

*Hint:* Use a generator expression inside [`sum`](https://docs.python.org/3/library/functions.html#sum).


In [ ]:
### YOUR CODE HERE ###
def count_papers(min_year: int = 0) -> int:
    ...


In [ ]:
def test_count_papers():
    assert count_papers() == 5, f"Expected 5, got {count_papers()}"
    assert count_papers(min_year=2020) == 3, f"Expected 3, got {count_papers(min_year=2020)}"
    assert count_papers(min_year=2025) == 0, f"Expected 0, got {count_papers(min_year=2025)}"
    print("PASS: count_papers")

test_count_papers()


**Q1 b) Write the JSON schema `count_papers_schema` for this tool.** The `name` must match, the `description` must be specific enough that a model knows when to reach for it, and `min_year` must NOT be in `required` (it has a default).

*Hint:* Follow the schema pattern from [Anthropic tool use docs](https://docs.anthropic.com/en/docs/build-with-claude/tool-use).


In [ ]:
### YOUR CODE HERE ###
count_papers_schema = {
    ...
}


In [ ]:
def test_count_papers_schema():
    assert count_papers_schema["name"] == "count_papers"
    assert len(count_papers_schema["description"]) > 10, "Description must be specific"
    assert count_papers_schema["input_schema"]["type"] == "object"
    assert "min_year" in count_papers_schema["input_schema"]["properties"]
    assert count_papers_schema["input_schema"]["properties"]["min_year"]["type"] == "integer"
    assert "min_year" not in count_papers_schema["input_schema"].get("required", [])
    print("PASS: count_papers_schema")

test_count_papers_schema()


***
<a id='filter-by-author'></a>
## 3. `filter_by_author` - function + schema

Papers have multiple authors. Write a tool that returns paper IDs where a given name appears (case-insensitive substring match). Empty list on no match - never raise.


**Q2 a) Implement `filter_by_author(author_name: str) -> List[str]`. Match case-insensitively and as a substring against each author's name.**

*Hint:* [`str.lower`](https://docs.python.org/3/library/stdtypes.html#str.lower) on both sides of the comparison, [`any`](https://docs.python.org/3/library/functions.html#any) across the author list.


In [ ]:
### YOUR CODE HERE ###
def filter_by_author(author_name: str) -> List[str]:
    ...


In [ ]:
def test_filter_by_author():
    assert filter_by_author("Vaswani") == ["2501.11111"]
    assert filter_by_author("VASWANI") == ["2501.11111"], "should be case-insensitive"
    assert filter_by_author("NoSuchPerson") == [], "empty list, not exception"
    # substring match: 'e' appears in Devlin, Chang, Perez, ...
    assert len(filter_by_author("e")) >= 2, "substring match should find multiple"
    print("PASS: filter_by_author")

test_filter_by_author()


**Q2 b) Write `filter_by_author_schema` for this tool.** `author_name` is required (no sensible default).


In [ ]:
### YOUR CODE HERE ###
filter_by_author_schema = {
    ...
}


In [ ]:
def test_filter_by_author_schema():
    assert filter_by_author_schema["name"] == "filter_by_author"
    assert filter_by_author_schema["input_schema"]["properties"]["author_name"]["type"] == "string"
    assert "author_name" in filter_by_author_schema["input_schema"]["required"]
    print("PASS: filter_by_author_schema")

test_filter_by_author_schema()


***
<a id='dispatcher'></a>
## 4. Extending the dispatcher

You now have three tools: `count_papers`, `filter_by_author`, plus an existing `extract_info` (defined below). Build a dispatcher that (1) looks up the function by name, (2) invokes it with the args, (3) normalizes the return value (list -> comma-separated, dict -> JSON, otherwise -> `str(...)`), and (4) raises `KeyError` with a helpful message when the name is unknown.


In [ ]:
def extract_info(paper_id: str) -> str:
    if paper_id in FAKE_PAPERS:
        return json.dumps(FAKE_PAPERS[paper_id], indent=2)
    return f"There's no saved information related to paper {paper_id}."


**Q3 a) Register the three tools in `tool_functions` and implement `execute_tool`.** Follow the normalization rules from Notebook 01, Part 3.


In [ ]:
### YOUR CODE HERE ###
tool_functions = {
    ...
}

def execute_tool(tool_name: str, tool_args: dict) -> str:
    ...


In [ ]:
def test_dispatcher():
    assert execute_tool("count_papers", {}) == "5"
    assert execute_tool("filter_by_author", {"author_name": "Vaswani"}) == "2501.11111"

    result = execute_tool("extract_info", {"paper_id": "2501.11111"})
    data = json.loads(result)
    assert data["title"] == "Attention is All You Need"

    try:
        execute_tool("nonexistent", {})
        raise AssertionError("should have raised KeyError")
    except KeyError:
        pass

    print("PASS: dispatcher")

test_dispatcher()


***
<a id='defensive-dispatch'></a>
## 5. Defensive dispatch

The model sometimes constructs bad arguments - a string where an int was expected, an unknown keyword, a missing required field. If the dispatcher raises, the whole conversation dies. If it returns a *string* starting with `"Error"`, the model can read the error and try again.


**Q4) Implement `safe_execute_tool(tool_name, tool_args) -> str` that wraps `execute_tool` with the following contract:**

- Success -> return whatever `execute_tool` returns.
- Unknown tool (`KeyError`) -> return `"Error: unknown tool 'X'. Available tools: ..."`.
- Bad args (`TypeError`) -> return `"Error calling X: <message>"`.
- Anything else -> return `"Error calling X: <ExceptionType>: <message>"`.

*Hint:* [Python exception hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy) - catch specific exceptions before the general `Exception` base class.


In [ ]:
### YOUR CODE HERE ###
def safe_execute_tool(tool_name: str, tool_args: dict) -> str:
    ...


In [ ]:
def test_safe_execute_tool():
    assert safe_execute_tool("count_papers", {}) == "5"

    r = safe_execute_tool("delete_everything", {})
    assert r.startswith("Error"), r
    assert "delete_everything" in r
    assert "count_papers" in r, "error should list available tools"

    r = safe_execute_tool("count_papers", {"nonsense_arg": 42})
    assert r.startswith("Error"), r

    r = safe_execute_tool("filter_by_author", {})
    assert r.startswith("Error"), r

    print("PASS: safe_execute_tool")

test_safe_execute_tool()


***
<a id='generalization'></a>
## 6. Generalization - wrap a real API

The previous questions used the fake in-memory store. This question asks you to transfer the same design to a different domain: pick a real API you know (weather, GitHub search, currency conversion, dictionary lookup, etc.), and wrap ONE endpoint of it as an MCP tool.

**Q5) Write three things:**

1. A Python function that wraps one endpoint of your chosen API. Handle failure with a returned string, not a raise. If you do not have internet access, mock the API call and mark it with `# TODO: replace with real HTTP call`.
2. A JSON schema for the tool.
3. A short markdown cell (2-4 sentences) explaining what you deliberately exposed and what you deliberately did NOT expose (and why).

*Hint:* [`requests`](https://requests.readthedocs.io/en/latest/) or [`httpx`](https://www.python-httpx.org/) for HTTP calls. Set a timeout on every request.


In [ ]:
### YOUR CODE HERE ###
# 1. Your function
def my_tool(...):
    ...

# 2. Your schema
my_tool_schema = {
    ...
}


*ANSWER HERE*

(Explain your design choices in 2-4 sentences: what did you expose, what did you deliberately not expose, and why?)
